In [2]:
############################################################
# 0. Colab Project Folder Setup
############################################################

# For Google Colab only: upload the CSV into data/ before running the analysis.
# When running from a local clone of this repository, skip this cell because the CSV is already in data/.

# Create the data folder used by the replication project.
!mkdir -p data

# Create the output/tables folder used to save the final replication table.
!mkdir -p output/tables

# Import the file-upload tool from Google Colab.
from google.colab import files

# Open a file picker so the user can upload the required CSV dataset.
uploaded = files.upload()

# Import os so Python can work with files and folders.
import os

# Set the expected filename of the replication dataset.
uploaded_file_name = "modified_hansen_data.csv"

# Move the uploaded CSV file into the project's data folder if it is not already there.
if os.path.exists(uploaded_file_name):
    os.replace(uploaded_file_name, f"data/{uploaded_file_name}")

# Confirm that the dataset is in the expected project folder.
print("Data folder contents:", os.listdir("data"))

############################################################
# 1. Replication Information
############################################################

# This is a reduced replication of Hansen (2015), Table 2.
# The modified dataset contains male, white, aged, and acc.
# It does not contain prior BAC tests or PBT.

############################################################
# 2. Load Packages
############################################################

# Import pandas for loading, cleaning, and organizing the dataset.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import statsmodels for running regression models.
import statsmodels.api as sm

# Import sys to report the Python version.
import sys

# Print the Python version used to run the code.
print("Python version:", sys.version)

# Print the pandas version used to run the code.
print("pandas version:", pd.__version__)

# Print the numpy version used to run the code.
print("numpy version:", np.__version__)

############################################################
# 3. Set Reproducible File Paths
############################################################

# Set the CSV file path relative to the main GitHub repository folder.
file_path = "data/modified_hansen_data.csv"

# Set the output file path relative to the main GitHub repository folder.
output_path = "output/tables/reduced_table2_replication.csv"

############################################################
# 4. Load Dataset
############################################################

# Load the CSV file into a pandas DataFrame.
df = pd.read_csv(file_path)

# Save the number of rows in the raw dataset before cleaning.
original_rows = len(df)

# Standardize all column names by making them lowercase and removing extra spaces.
df.columns = df.columns.str.lower().str.strip()

############################################################
# 5. Construct BAC Running Variable
############################################################

# If low_score does not exist, create it using the lower of the two BAC readings.
if "low_score" not in df.columns:
    df["low_score"] = df[["alcohol1", "alcohol2"]].min(axis=1)

# Convert low_score to numeric values.
df["low_score"] = pd.to_numeric(df["low_score"], errors="coerce")

# Convert low_score from BAC points to decimal BAC if needed.
if df["low_score"].max() > 1:
    df["bac"] = df["low_score"] / 1000
else:
    df["bac"] = df["low_score"]

# Create BAC bins rounded to 0.001 for clustered standard errors.
df["bac_bin"] = df["bac"].round(3)

############################################################
# 6. Clean Data and Apply Sample Restrictions
############################################################

# List the variables needed for the reduced Table 2 replication.
required_vars = ["bac", "low_score", "male", "white", "aged", "acc"]

# Add year to the required variables if it exists in the dataset.
if "year" in df.columns:
    required_vars.append("year")

# Convert all required variables to numeric values.
for var in required_vars:
    df[var] = pd.to_numeric(df[var], errors="coerce")

# Drop observations with missing values in the required variables.
df = df.dropna(subset=required_vars)

# Restrict the sample to 1999–2007 if the year variable exists.
if "year" in df.columns:
    df = df[(df["year"] >= 1999) & (df["year"] <= 2007)]

# Restrict the sample to drivers age 21 and older.
df = df[df["aged"] >= 21]

############################################################
# 7. Construct Treatment Indicators
############################################################

# Create DUI treatment based on BAC greater than or equal to 0.08.
df["dui"] = (df["bac"] >= 0.08).astype(int)

# Create aggravated DUI treatment based on BAC greater than or equal to 0.15.
df["agg_dui"] = (df["bac"] >= 0.15).astype(int)

############################################################
# 8. Print Data Checks
############################################################

# Print the number of rows in the raw dataset.
print("Original rows:", original_rows)

# Print the number of rows after cleaning and sample restrictions.
print("Rows after cleaning:", len(df))

# Print the minimum and maximum BAC values after cleaning.
print("BAC range:", df["bac"].min(), df["bac"].max())

############################################################
# 9. Define Outcome Variables
############################################################

# Define the predetermined outcome variables available in the modified dataset.
outcomes = {
    "Male": "male",
    "White": "white",
    "Age": "aged",
    "Accident": "acc"
}

############################################################
# 10. Define Regression Discontinuity Function
############################################################

# Define a function that runs one local linear RDD regression.
def run_rdd(data, outcome, cutoff, treatment_var, bandwidth=0.05):
    # Keep only observations within the bandwidth around the cutoff.
    temp = data[(data["bac"] >= cutoff - bandwidth) & (data["bac"] <= cutoff + bandwidth)].copy()

    # Create the centered running variable.
    temp["bac_centered"] = temp["bac"] - cutoff

    # Create the interaction between treatment status and the centered running variable.
    temp["interaction"] = temp[treatment_var] * temp["bac_centered"]

    # Define the regression outcome variable.
    y = temp[outcome]

    # Define the regression explanatory variables.
    X = temp[[treatment_var, "bac_centered", "interaction"]]

    # Add a constant term to the regression.
    X = sm.add_constant(X)

    # Estimate the OLS regression with clustered standard errors by BAC bin.
    model = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": temp["bac_bin"]})

    # Extract the coefficient on the treatment variable.
    coef = model.params[treatment_var]

    # Extract the clustered standard error for the treatment variable.
    se = model.bse[treatment_var]

    # Calculate the mean of the outcome just below the cutoff.
    mean_below = temp[temp["bac"] < cutoff][outcome].mean()

    # Count the number of observations used in the regression.
    nobs = int(model.nobs)

    # Return the regression results.
    return coef, se, mean_below, nobs

############################################################
# 11. Estimate RDD Models for Table 2
############################################################

# Create an empty list to store the final table rows.
results = []

# Loop through each predetermined characteristic.
for label, var in outcomes.items():
    # Run the RDD regression at the 0.08 DUI threshold.
    dui_coef, dui_se, dui_mean, dui_n = run_rdd(df, var, 0.08, "dui")

    # Run the RDD regression at the 0.15 aggravated DUI threshold.
    agg_coef, agg_se, agg_mean, agg_n = run_rdd(df, var, 0.15, "agg_dui")

    # Add the results for this outcome variable to the results list.
    results.append({
        "Characteristic": label,
        "DUI Coef.": dui_coef,
        "DUI SE": dui_se,
        "DUI Mean Below Cutoff": dui_mean,
        "DUI Observations": dui_n,
        "Agg. DUI Coef.": agg_coef,
        "Agg. DUI SE": agg_se,
        "Agg. DUI Mean Below Cutoff": agg_mean,
        "Agg. DUI Observations": agg_n
    })

############################################################
# 12. Create Final Replication Table
############################################################

# Convert the results list into a DataFrame.
table2_replication = pd.DataFrame(results)

# Round the numerical results for readability.
table2_replication = table2_replication.round(4)

# Print a note explaining what the table replicates.
print("Reduced replication of Hansen (2015), Table 2.")

# Print a note explaining why this is reduced.
print("This version includes only variables available in the modified dataset: male, white, age, and accident.")

# Print a note explaining which original variables are missing.
print("The original paper also includes prior BAC tests and PBT, but those variables are not available here.")

# Display the final replication table.
print(table2_replication)

############################################################
# 13. Export Final Table
############################################################

# Export the final replication table to the output/tables folder.
table2_replication.to_csv(output_path, index=False)

# Print the saved output location.
print("Table saved to:", output_path)

Saving modified_hansen_data.csv to modified_hansen_data.csv
Data folder contents: ['modified_hansen_data.csv']
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
pandas version: 2.2.2
numpy version: 2.0.2
Original rows: 214558
Rows after cleaning: 214558
BAC range: 0.0 0.435
Reduced replication of Hansen (2015), Table 2.
This version includes only variables available in the modified dataset: male, white, age, and accident.
The original paper also includes prior BAC tests and PBT, but those variables are not available here.
  Characteristic  DUI Coef.  DUI SE  DUI Mean Below Cutoff  DUI Observations  \
0           Male     0.0061  0.0051                 0.7876             93899   
1          White     0.0017  0.0055                 0.8469             93899   
2            Age    -0.1470  0.1658                34.9398             93899   
3       Accident    -0.0008  0.0035                 0.0994             93899   

   Agg. DUI Coef.  Agg. DUI SE  Agg. DUI Mean Below Cu

In [3]:
# Import the download tool from Google Colab.
from google.colab import files

# Download the replication table CSV to your computer.
files.download("output/tables/reduced_table2_replication.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>